<a href="https://colab.research.google.com/github/Arfwjn/Speech-to-Text/blob/main/STT_WhisperONNX.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Sel 1: Instalasi Library
!pip uninstall -y datasets transformers torchcodec torch torchvision torchaudio --quiet

# 1. Instal Torch + companion libraries
print("--- Menginstal torch, torchvision, torchaudio ---")
!pip install torch torchvision torchaudio --no-cache-dir --quiet

# 2. Instal sisa library yang diperlukan
print("--- Menginstal datasets, transformers, peft, dll. ---")
!pip install "datasets[audio]>=2.17.0" --quiet
!pip install transformers accelerate evaluate jiwer librosa soundfile --quiet
!pip install bitsandbytes peft --quiet

print("--- Instalasi Selesai ---")

--- Menginstal torch, torchvision, torchaudio ---
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 899.7/899.7 MB 219.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.3/594.3 MB 182.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 187.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 151.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 954.8/954.8 kB 328.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.1/193.1 MB 212.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 160.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.6/63.6 MB 218.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.5/267.5 MB 135.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 288.2/288.2 MB 269.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.3/39.3 MB 194.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.0/90.0 kB 205.0 MB/

In [ ]:
# Sel 2: Import & Pengaturan Global
import torch
import evaluate
import jiwer
from datasets import load_dataset, DatasetDict, Audio
from transformers import (
    WhisperFeatureExtractor,
    WhisperTokenizer,
    WhisperProcessor,
    WhisperForConditionalGeneration,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer
)
from transformers.models.whisper.english_normalizer import EnglishTextNormalizer
from peft import LoraConfig, PeftModel, get_peft_model, prepare_model_for_kbit_training
import dataclasses
from dataclasses import dataclass
from typing import Any, Dict, List, Union

# PENGATURAN
DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
MODEL_NAME = "openai/whisper-small.en"
NEW_MODEL_NAME = "whisper-small-en-finetuned-lora"
LANGUAGE = "en"
TASK = "transcribe"
NUM_TRAIN_SAMPLES = 1000   # Batasi 5000 sampel untuk training cepat (dari train.100)
NUM_EVAL_SAMPLES = 200     # Jumlah sampel evaluasi

print(f"Menggunakan perangkat: {DEVICE}")

Menggunakan perangkat: cuda:0


In [ ]:
# Sel 3: Muat Processor & Tokenizer
print("Memuat Processor (Feature Extractor + Tokenizer)")
feature_extractor = WhisperFeatureExtractor.from_pretrained(MODEL_NAME)
tokenizer = WhisperTokenizer.from_pretrained(MODEL_NAME, language=LANGUAGE, task=TASK)
processor = WhisperProcessor.from_pretrained(MODEL_NAME, language=LANGUAGE, task=TASK)
print("Processor berhasil dimuat.")

Memuat Processor (Feature Extractor + Tokenizer)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/805 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

merges.txt: 0.00B [00:00, ?B/s]

normalizer.json: 0.00B [00:00, ?B/s]

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

Processor berhasil dimuat.


In [ ]:
# Sel 4: Muat dan Siapkan Dataset
print("Memuat dan menyiapkan dataset...")

# 1. Buat DatasetDict
raw_datasets = DatasetDict()

# 2. Muat 'train.100' untuk training
raw_datasets["train"] = load_dataset("librispeech_asr", "clean", split="train.100")
raw_datasets["train"] = raw_datasets["train"].select(range(NUM_TRAIN_SAMPLES))

# 3. Muat 'validation' untuk evaluasi
raw_datasets["eval"] = load_dataset("librispeech_asr", "clean", split="validation")
raw_datasets["eval"] = raw_datasets["eval"].select(range(NUM_EVAL_SAMPLES))

# 4. Pastikan audio di-resample ke 16kHz
raw_datasets = raw_datasets.cast_column("audio", Audio(sampling_rate=16000))

print(f"Dataset dimuat:\n{raw_datasets}")

# 5. Buat fungsi pre-processing
def prepare_dataset(batch):
    # Proses audio
    audio = batch["audio"]
    batch["input_features"] = feature_extractor(audio["array"], sampling_rate=audio["sampling_rate"]).input_features[0]

    # Proses teks (transkrip)
    batch["labels"] = tokenizer(batch["text"]).input_ids
    return batch

# 6. Terapkan fungsi ke dataset
print("\nMemproses dataset (mapping)...")
tokenized_datasets = raw_datasets.map(
    prepare_dataset,
    remove_columns=raw_datasets.column_names["train"]
)
print("Dataset selesai diproses.")

Memuat dan menyiapkan dataset...


README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/48 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/48 [00:00<?, ?it/s]

clean/test/0000.parquet:   0%|          | 0.00/350M [00:00<?, ?B/s]

...

Resolving data files:   0%|          | 0/48 [00:00<?, ?it/s]

Dataset dimuat:
DatasetDict({
    train: Dataset({
        features: ['file', 'audio', 'text', 'speaker_id', 'chapter_id', 'id'],
        num_rows: 1000
    })
    eval: Dataset({
        features: ['file', 'audio', 'text', 'speaker_id', 'chapter_id', 'id'],
        num_rows: 200
    })
})

Memproses dataset (mapping)...


Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Dataset selesai diproses.


In [ ]:
# Sel 5 (Bypass API)

import re

# 1. Definisikan Data Collator
@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        # Pisahkan input_features dan labels
        input_features = [{"input_features": feature["input_features"]} for feature in features]
        label_features = [{"input_ids": feature["labels"]} for feature in features]

        # Gabungkan (batch) features audio
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        # Gabungkan (batch) features teks
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        # Ganti padding -100 agar diabaikan oleh loss function
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)

        # Jika ada 'bos_token_id', hapus dari awal labels
        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().item():
            labels = labels[:, 1:]

        batch["labels"] = labels
        return batch

# Inisialisasi Data Collator
data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)
print("Data Collator siap.")

# 2. Definisikan Metrik Evaluasi (WER)
metric = evaluate.load("wer")

# Normalizer Manual
def simple_normalizer(text):
    if text is None:
        return ""
    text = text.lower()  # Ubah ke lowercase
    text = re.sub(r"[^\w\s]", "", text)  # Hapus semua tanda baca
    text = re.sub(r"\s+", " ", text).strip()  # Ganti spasi ganda -> tunggal & rapikan
    return text

normalizer = simple_normalizer

print("Metrik WER (dengan normalisasi MANUAL) siap.")

def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids

    # Ganti -100 (padding) dengan pad_token_id
    label_ids[label_ids == -100] = tokenizer.pad_token_id

    # Decode hasil prediksi dan referensi
    pred_str = tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = tokenizer.batch_decode(label_ids, skip_special_tokens=True)

    # Normalisasi teks menggunakan fungsi manual kita
    pred_str_norm = [normalizer(s) for s in pred_str]
    label_str_norm = [normalizer(s) for s in label_str]

    # Hitung WER
    wer = 100 * metric.compute(predictions=pred_str_norm, references=label_str_norm)
    return {"wer": wer}

Data Collator siap.


Metrik WER (dengan normalisasi MANUAL) siap.


In [ ]:
# Sel 6: Muat Model & Konfigurasi LoRA
print("Memuat base model (8-bit) dan menerapkan LoRA...")

# 1. Muat base model dalam 8-bit
model = WhisperForConditionalGeneration.from_pretrained(MODEL_NAME, load_in_8bit=True, device_map="auto")

# 2. Siapkan model untuk training 8-bit
model = prepare_model_for_kbit_training(model)

# 3. Konfigurasi LoRA
config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none"
)

# 4. Terapkan LoRA ke model
model = get_peft_model(model, config)

# 5. Tampilkan jumlah parameter yang akan dilatih
model.print_trainable_parameters()

Memuat base model (8-bit) dan menerapkan LoRA...


config.json: 0.00B [00:00, ?B/s]

The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.


model.safetensors:   0%|          | 0.00/151M [00:00<?, ?B/s]

generation_config.json: 0.00B [00:00, ?B/s]

trainable params: 147,456 || all params: 37,907,712 || trainable%: 0.3890


In [ ]:
# Sel 7: Konfigurasi Training
print("Menyiapkan Training Arguments...")

training_args = Seq2SeqTrainingArguments(
    output_dir=NEW_MODEL_NAME,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,  # Efektif batch size = 8 * 2 = 16
    learning_rate=1e-5,
    warmup_steps=50,
    num_train_epochs=3,
    eval_strategy="steps",
    eval_steps=500,
    save_strategy="steps",
    save_steps=500,
    logging_steps=50,
    per_device_eval_batch_size=8,
    predict_with_generate=True,
    generation_max_length=225,
    fp16=True,
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
    push_to_hub=False,
    report_to=["none"],
)

Menyiapkan Training Arguments...


In [ ]:
# Sel 8: Inisialisasi & Mulai Training

# 1. Inisialisasi Trainer
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["eval"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    tokenizer=processor.feature_extractor,
)

# 2. Simpan processor agar bisa dipakai nanti
processor.save_pretrained(training_args.output_dir)

# 3. Mulai Training!
print("\n" + "="*30)
print("     MEMULAI FINE-TUNING     ")
print("="*30 + "\n")

trainer.train()

print("\n\n" + "="*30)
print("     FINE-TUNING SELESAI     ")
print("="*30 + "\n")

# 4. Simpan adapter LoRA yang sudah dilatih
print(f"Menyimpan adapter LoRA ke ./{NEW_MODEL_NAME}")
model.save_pretrained(NEW_MODEL_NAME)

/tmp/ipython-input-2841987890.py:5: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(



     MEMULAI FINE-TUNING     



You're using a WhisperTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1044: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/utils/checkpoint.py:85: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:181: UserWarning: MatMul8bitLt: input

Step,Training Loss,Validation Loss




     FINE-TUNING SELESAI     

Menyimpan adapter LoRA ke ./whisper-tiny-en-finetuned-lora


In [ ]:
# Sel 9: Evaluasi Model yang Sudah di-Fine-Tuned
from transformers import pipeline, WhisperForConditionalGeneration
from peft import PeftModel
from tqdm import tqdm

print("Memuat pipeline dengan model yang sudah di-fine-tuned...")

# 1. Muat base model (bisa 8-bit lagi untuk inferensi cepat)
base_model = WhisperForConditionalGeneration.from_pretrained(
    MODEL_NAME, load_in_8bit=True, device_map="auto"
)

# 2. Terapkan adapter LoRA yang sudah dilatih
ft_model = PeftModel.from_pretrained(base_model, NEW_MODEL_NAME)

# 3. Muat processor yang tadi disimpan
processor = WhisperProcessor.from_pretrained(NEW_MODEL_NAME)

# 4. Buat pipeline BARU dengan model hasil fine-tuning
pipe_ft = pipeline(
    "automatic-speech-recognition",
    model=ft_model,
    tokenizer=processor.tokenizer,
    feature_extractor=processor.feature_extractor,
    chunk_length_s=30
)
print("Pipeline fine-tuned (FT) berhasil dimuat.")


# 5. Jalankan evaluasi
print(f"\nMemulai evaluasi FT pada {NUM_EVAL_SAMPLES} sampel audio...")
references = []
hypotheses = []
eval_dataset_audio = raw_datasets["eval"] # Ambil dari dataset mentah (sebelum di-map)

for i in tqdm(range(len(eval_dataset_audio))):
    sample = eval_dataset_audio[i]
    audio_input = sample["audio"]["array"]
    reference_text = sample["text"]

    # Prediksi menggunakan model fine-tuned
    prediction = pipe_ft(audio_input.copy())
    predicted_text = prediction["text"]

    # Gunakan normalisasi yang sama (manual) dengan saat training
    references.append(normalizer(reference_text))
    hypotheses.append(normalizer(predicted_text))

# 6. Hitung dan Tampilkan Hasil Akhir
print("\nEvaluasi FT selesai. Menghitung hasil...")
wer_ft = jiwer.wer(references, hypotheses)
accuracy_ft = (1 - wer_ft) * 100

print("\n" + "-" * 50)
print(f"HASIL EVALUASI MODEL FINE-TUNED: {NEW_MODEL_NAME}   ")
print(f"(Berdasarkan {len(references)} sampel)      ")
print("-" * 50)
print(f"Word Error Rate (WER): {wer_ft:.4f}  (atau {wer_ft:.2%})")
print(f"Perkiraan Akurasi: {accuracy_ft:.2f}%")
print("-" * 50)

target_akurasi = 90.0
if accuracy_ft >= target_akurasi:
    print(f"KESIMPULAN: Model ini MEMENUHI target akurasi proyek (≥ {target_akurasi}%)!")
else:
    print(f"KESIMPULAN: Model ini BELUM MEMENUHI target akurasi proyek (≥ {target_akurasi}%)!")

Memuat pipeline dengan model yang sudah di-fine-tuned...


The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.
Device set to use cuda:0
Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on it's own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcription).


Pipeline fine-tuned (FT) berhasil dimuat.

Memulai evaluasi FT pada 200 sampel audio...


100%|██████████| 200/200 [03:02<00:00,  1.09it/s]


Evaluasi FT selesai. Menghitung hasil...

--------------------------------------------------
   HASIL EVALUASI MODEL FINE-TUNED: whisper-tiny-en-finetuned-lora   
      (Berdasarkan 200 sampel)      
--------------------------------------------------
Word Error Rate (WER): 0.0578  (atau 5.78%)
Perkiraan Akurasi: 94.22%
--------------------------------------------------
 KESIMPULAN: Model ini MEMENUHI target akurasi proyek (≥ 90.0%)!


In [ ]:
# Sel 10 (Instalasi Deployment)
print("--- Menginstal library untuk DEPLOYMENT (ONNX, Peft, Gradio) ---")

# 1. Instal Gradio & Librosa
!pip install gradio librosa

# 2. Upgrade stack utama (menghapus versi training)
!pip install --upgrade pip
!pip install --upgrade torch transformers optimum[onnxruntime-gpu] huggingface_hub

# 3. Instal PEFT
!pip install peft

# 4. Perbaikan Protobuf (untuk menghindari error 'GetPrototype'/'AttributeError')
!pip install --upgrade protobuf tensorboard

print("--- Instalasi Deployment Selesai ---")

--- Menginstal library untuk DEPLOYMENT (ONNX, Peft, Gradio) ---
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 75.3 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 136.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 133.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 300.5/300.5 MB 70.1 MB/s  0:00:03
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 92.0 MB/s  0:00:00
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.1
    Uninstalling tokenizers-0.22.1:
      Successfully uninstalled tokenizers-0.22.1
  Attempting uninstall: transformers
    Found existing installation: transformers 4.57.1
    Uninstalling transformers-4.57.1:
      Successfully uninstalled transformers-4.57.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8/8 [optimum-on

--- Instalasi Deployment Selesai ---


In [ ]:
# Sel 11

import torch

# PENGATURAN
DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
MODEL_NAME = "openai/whisper-small.en"
NEW_MODEL_NAME = "whisper-small-en-finetuned-lora" # Path ke model Anda
LANGUAGE = "en"
TASK = "transcribe"

print(f"Variabel berhasil di-set. Menggunakan perangkat: {DEVICE}")
print(f"Base model: {MODEL_NAME}")
print(f"Adapter LoRA: {NEW_MODEL_NAME}")

Variabel berhasil di-set. Menggunakan perangkat: cuda:0
Base model: openai/whisper-tiny.en
Adapter LoRA: whisper-tiny-en-finetuned-lora


In [ ]:
# Sel 12
# JALANKAN SETELAH SEL 11

import torch
from transformers import WhisperForConditionalGeneration, WhisperProcessor
from transformers import pipeline
from peft import PeftModel
import os
import gradio as gr
import librosa
import numpy as np
from optimum.onnxruntime import ORTModelForSpeechSeq2Seq

# --- Bagian 1: MERGE & EKSPOR ---
print("--- Memulai Proses Merge & Ekspor ONNX ---")

# 1. Cek apakah variabel dari Sel 11 sudah ada
try:
    BASE_MODEL = MODEL_NAME
    LORA_ADAPTER = NEW_MODEL_NAME
    DEVICE = DEVICE
except NameError:
    print("="*50)
    print("ERROR: Variabel (MODEL_NAME/NEW_MODEL_NAME/DEVICE) tidak ada.")
    print("Harap JALANKAN SEL 10.5 (Pengganti Sel 2) Anda terlebih dahulu.")
    print("="*50)
    raise

# 2. Tentukan Path Output
MERGED_MODEL_DIR = "./whisper-small-en-merged"
ONNX_MODEL_DIR = "./whisper-small-en-onnx"

# 3. Muat & Gabungkan Model
print(f"Memuat base model {BASE_MODEL} dalam float16...")
base_model_ft16 = WhisperForConditionalGeneration.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
    device_map="auto"
)
print(f"Memuat adapter LoRA dari {LORA_ADAPTER}...")
merged_model = PeftModel.from_pretrained(base_model_ft16, LORA_ADAPTER)
print("Menggabungkan adapter (merge and unload)...")
merged_model = merged_model.merge_and_unload()

# 4. Simpan Model & Processor yang sudah digabung
print(f"Menyimpan model & processor yang digabung ke {MERGED_MODEL_DIR}...")
merged_model.save_pretrained(MERGED_MODEL_DIR)
processor = WhisperProcessor.from_pretrained(LORA_ADAPTER)
processor.save_pretrained(MERGED_MODEL_DIR)
print("--- Proses Merge Selesai. ---")

# 5. Ekspor ke ONNX
print(f"\n--- Menjalankan Ekspor ke ONNX ---")
!optimum-cli export onnx --model {MERGED_MODEL_DIR} --task automatic-speech-recognition {ONNX_MODEL_DIR}
print(f"--- Ekspor ONNX Selesai. Model ada di {ONNX_MODEL_DIR} ---")


# --- Bagian 2: JALANKAN GRADIO ---
print("\n--- Memulai Interface Demo Gradio ---")

ONNX_PATH = ONNX_MODEL_DIR
MERGED_PATH = MERGED_MODEL_DIR

print(f"Memuat model ONNX dari: {ONNX_PATH}")
print(f"Memuat processor dari: {MERGED_PATH}")

# 1. Muat Model ONNX dan BUAT PIPELINE
try:
    onnx_processor = WhisperProcessor.from_pretrained(MERGED_PATH)
    onnx_model = ORTModelForSpeechSeq2Seq.from_pretrained(
        ONNX_PATH,
        provider="CUDAExecutionProvider" if DEVICE == "cuda:0" else "CPUExecutionProvider",
        use_cache=False
    )
    print("--- Model dan Processor Berhasil Dimuat ---")

    # --- PERBAIKANNYA DI SINI ---
    # Kita buat pipeline yang akan menangani chunking audio panjang
    print("--- Membuat pipeline ASR... ---")
    onnx_pipeline = pipeline(
        "automatic-speech-recognition",
        model=onnx_model,
        tokenizer=onnx_processor.tokenizer,
        feature_extractor=onnx_processor.feature_extractor,
        device=DEVICE # Arahkan pipeline untuk menggunakan GPU
    )
    print("--- Pipeline ASR (untuk audio panjang) siap ---")

except Exception as e:
    print(f"GAGAL MEMUAT MODEL/PIPELINE: {e}")
    raise

# 2. Buat Fungsi Inti (Transkripsi)
def transcribe_audio(audio_input):
    if audio_input is None:
        return "Tidak ada audio terdeteksi. Silakan upload file."

    print("Menerima audio...")
    sample_rate, data = audio_input
    data = data.astype(np.float32) / 32768.0

    # Resample dan mono
    if data.ndim > 1:
        data = np.mean(data, axis=1)
    if sample_rate != 16000:
        data = librosa.resample(y=data, orig_sr=sample_rate, target_sr=16000)

    print("Audio selesai diproses, menjalankan inferensi pipeline...")

    try:
        result = onnx_pipeline(data)
        transcription = result["text"]

        print(f"Hasil Transkripsi: {transcription}")
        return transcription
    except Exception as e:
        print(f"Error saat inferensi: {e}")
        return f"Error: {e}"

# 3. Luncurkan Interface
print("Meluncurkan interface Gradio... Klik link publik di bawah.")
iface = gr.Interface(
    fn=transcribe_audio,
    inputs=gr.Audio(type="numpy", label="Upload Audio Anda (WAV, MP3, M4A, dll.)"),
    outputs=gr.Textbox(label="Hasil Transkripsi"),
    title="🎧 Demo Model Whisper (Fine-Tuned & ONNX)",
    description="Interface ini menjalankan model Whisper yang sudah di-fine-tune dan dikonversi ke ONNX."
)
iface.launch(share=True, debug=True)

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

...

/usr/local/lib/python3.12/dist-packages/transformers/modeling_utils.py:3922: UserWarning: Moving the following attributes in the config to the generation config: {'max_length': 448, 'suppress_tokens': [1, 2, 7, 8, 9, 10, 14, 25, 26, 27, 28, 29, 31, 58, 59, 60, 61, 62, 63, 90, 91, 92, 93, 357, 366, 438, 532, 685, 705, 796, 930, 1058, 1220, 1267, 1279, 1303, 1343, 1377, 1391, 1635, 1782, 1875, 2162, 2361, 2488, 3467, 4008, 4211, 4600, 4808, 5299, 5855, 6329, 7203, 9609, 9959, 10563, 10786, 11420, 11709, 11907, 13163, 13697, 13700, 14808, 15306, 16410, 16791, 17992, 19203, 19510, 20724, 22305, 22935, 27007, 30109, 30420, 33409, 34949, 40283, 40493, 40549, 47282, 49146, 50257, 50357, 50358, 50359, 50360, 50361]}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(


--- Proses Merge Selesai. ---

--- Menjalankan Ekspor ke ONNX ---
2025-11-16 07:32:34.149900: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1763278354.170821   15462 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1763278354.177959   15462 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1763278354.196291   15462 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1763278354.196316   15462 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:176327835

Could not find any ONNX files with standard file name decoder_model_merged.onnx, files found: [PosixPath('encoder_model.onnx'), PosixPath('decoder_model.onnx')]. Please make sure to pass a `file_name` and/or `subfolder` argument to `from_pretrained` when loading an ONNX file with non-standard file names.
Device set to use cuda:0


--- Model dan Processor Berhasil Dimuat ---
--- Membuat pipeline ASR... ---
--- Pipeline ASR (untuk audio panjang) siap ---
Meluncurkan interface Gradio... Klik link publik di bawah.
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://9ff48528397b26808e.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Menerima audio...
Audio selesai diproses, menjalankan inferensi pipeline...
Error saat inferensi: You have passed more than 3000 mel input features (> 30 seconds) which automatically enables long-form generation which requires the model to predict timestamp tokens. Please either pass `return_timestamps=True` or make sure to pass no more than 3000 mel input features.
Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://9ff48528397b26808e.gradio.live


### **Import to Drive**

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
## Salin Folder Model ke Google Drive

NEW_MODEL_NAME = "whisper-small-en"
MERGED_MODEL_DIR = f"./{NEW_MODEL_NAME}-merged"
ONNX_MODEL_DIR = f"./{NEW_MODEL_NAME}-onnx"

# Folder tujuan di Drive
TARGET_DRIVE_FOLDER = "/content/drive/MyDrive/CapstoneProject/ModelSTT-ONNX-GRADIO"

# Salin folder Merged (PyTorch final) dan folder ONNX
print(f"Menyalin folder {MERGED_MODEL_DIR} ke Drive")
!cp -r {MERGED_MODEL_DIR} {TARGET_DRIVE_FOLDER}/

print(f"Menyalin folder {ONNX_MODEL_DIR} ke Drive")
!cp -r {ONNX_MODEL_DIR} {TARGET_DRIVE_FOLDER}/

print("\n" + "="*50)
print(f"PENYALINAN SELESAI!")
print(f"Model Anda tersimpan di: {TARGET_DRIVE_FOLDER}")

Menyalin folder ./whisper-tiny-en-merged ke Drive
Menyalin folder ./whisper-tiny-en-onnx ke Drive

PENYALINAN SELESAI!
Model Anda tersimpan di: /content/drive/MyDrive/CapstoneProject/ModelSTT-ONNX-GRADIO
